In [3]:
pip install dash


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: C:\Users\afats\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import dash
from dash import dcc, html
import dash.dependencies as dd
import pandas as pd
import numpy as np
import plotly.express as px
from geopy.distance import geodesic

# Load dataset
df = pd.read_csv("C:/Users/afats/Downloads/fooddelivery.csv")

# Rename columns for easier access
df.columns = ['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 
              'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 
              'Delivery_location_longitude', 'Type_of_order', 'Type_of_vehicle', 'Time_taken(min)']

# Function to calculate Haversine distance
def calculate_distance(row):
    restaurant_coords = (row['Restaurant_latitude'], row['Restaurant_longitude'])
    delivery_coords = (row['Delivery_location_latitude'], row['Delivery_location_longitude'])
    return geodesic(restaurant_coords, delivery_coords).km

df['Distance_km'] = df.apply(calculate_distance, axis=1)

# Initialize Dash app
app = dash.Dash(__name__)

# Layout
app.layout = html.Div([
    html.H1("Food Delivery Dashboard", style={'textAlign': 'center'}),

    # Dropdown to filter by vehicle type
    html.Label("Select Vehicle Type:"),
    dcc.Dropdown(
        id='vehicle-type-dropdown',
        options=[{'label': v, 'value': v} for v in df['Type_of_vehicle'].unique()],
        value=None,
        placeholder="Select vehicle type",
        multi=False
    ),

    # Row 1: Histogram & Scatter Plot
    html.Div([
        dcc.Graph(id='delivery-time-hist', style={'width': '48%', 'display': 'inline-block'}),
        dcc.Graph(id='delivery-time-scatter', style={'width': '48%', 'display': 'inline-block'}),
    ]),

    # Row 2: Vehicle Type vs Delivery Time & Pie Chart
    html.Div([
        dcc.Graph(id='vehicle-time-bar', style={'width': '48%', 'display': 'inline-block'}),
        dcc.Graph(id='order-type-pie', style={'width': '48%', 'display': 'inline-block'}),
    ]),

    # Row 3: Heatmap
    html.Div([
        dcc.Graph(id='heatmap', style={'width': '100%'})
    ])
])

# Callbacks to update graphs
@app.callback(
    [dd.Output('delivery-time-hist', 'figure'),
     dd.Output('delivery-time-scatter', 'figure'),
     dd.Output('vehicle-time-bar', 'figure'),
     dd.Output('order-type-pie', 'figure'),
     dd.Output('heatmap', 'figure')],
    [dd.Input('vehicle-type-dropdown', 'value')]
)
def update_graphs(selected_vehicle):
    # Filter data based on dropdown
    filtered_df = df if selected_vehicle is None else df[df['Type_of_vehicle'] == selected_vehicle]

    # Histogram - Delivery Time
    hist_fig = px.histogram(filtered_df, x='Time_taken(min)', nbins=30, 
                            title="Delivery Time Distribution", color_discrete_sequence=['blue'])

    # Scatter Plot - Delivery Time vs Distance
    scatter_fig = px.scatter(filtered_df, x='Distance_km', y='Time_taken(min)', 
                             title="Delivery Time vs Distance", color_discrete_sequence=['red'])

    # Bar Chart - Average Delivery Time by Vehicle Type
    bar_fig = px.bar(df.groupby('Type_of_vehicle')['Time_taken(min)'].mean().reset_index(),
                     x='Type_of_vehicle', y='Time_taken(min)', title="Avg Delivery Time by Vehicle Type", 
                     color_discrete_sequence=['green'])

    # Pie Chart - Order Type Distribution
    pie_fig = px.pie(df, names='Type_of_order', title="Order Type Distribution", color_discrete_sequence=px.colors.qualitative.Set3)

    # Heatmap - Correlations
    heatmap_fig = px.imshow(df[['Delivery_person_Age', 'Delivery_person_Ratings', 'Time_taken(min)', 'Distance_km']].corr(),
                            color_continuous_scale='Viridis', title="Feature Correlation Heatmap")

    return hist_fig, scatter_fig, bar_fig, pie_fig, heatmap_fig

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)


Error on request:
Traceback (most recent call last):
  File "C:\Users\afats\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\werkzeug\serving.py", line 370, in run_wsgi
    execute(self.server.app)
  File "C:\Users\afats\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\werkzeug\serving.py", line 355, in execute
    data = self.rfile.read(10_000_000)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
MemoryError
